# 🛒 E-commerce ETL (DLT SQL + Change Data Feed)

This notebook implements a **modern CDC-based ETL pipeline** using:
- Delta Live Tables (DLT)
- SQL-first transformations
- Change Data Feed (CDF)
- AUTO CDC (`APPLY CHANGES INTO`)
- Streaming Tables
- Materialized Views

Architecture: **Bronze → Silver → Gold (CDC-driven)**

## Bronze Layer – Raw CDC-Enabled Streaming Tables

In [ ]:
CREATE OR REFRESH STREAMING TABLE bronze_orders_cdc
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  quality = 'bronze'
)
COMMENT 'Raw orders with Change Data Feed enabled'
AS
SELECT
  *,
  current_timestamp() AS ingestion_ts
FROM cloud_files(
  's3://my-bucket/raw/raw_orders.csv',
  'csv',
  map(
    'header', 'true',
    'inferSchema', 'true'
  )
);

In [ ]:
CREATE OR REFRESH STREAMING TABLE bronze_customers_cdc
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  quality = 'bronze'
)
COMMENT 'Raw customers with Change Data Feed enabled'
AS
SELECT
  *,
  current_timestamp() AS ingestion_ts
FROM cloud_files(
  's3://my-bucket/raw/raw_customers.csv',
  'csv',
  map(
    'header', 'true',
    'inferSchema', 'true'
  )
);

## Silver Layer – AUTO CDC (APPLY CHANGES)

In [ ]:
CREATE OR REFRESH STREAMING TABLE silver_orders
TBLPROPERTIES (
  quality = 'silver'
)
COMMENT 'Current-state completed orders maintained via AUTO CDC';

In [ ]:
APPLY CHANGES INTO silver_orders
FROM STREAM(bronze_orders_cdc)
KEYS (order_id)
APPLY AS DELETE WHEN order_status = 'CANCELLED'
SEQUENCE BY ingestion_ts
COLUMNS *
STORED AS SCD TYPE 1;

### Enrich Orders with Customers

In [ ]:
CREATE OR REFRESH STREAMING TABLE silver_orders_enriched
TBLPROPERTIES (
  quality = 'silver'
)
AS
SELECT
  o.order_id,
  o.customer_id,
  CAST(o.order_date AS DATE) AS order_date,
  o.product,
  o.quantity,
  o.unit_price,
  o.quantity * o.unit_price AS total_amount,
  c.customer_name,
  c.email,
  c.country,
  c.signup_date
FROM STREAM(silver_orders) o
LEFT JOIN STREAM(bronze_customers_cdc) c
  ON o.customer_id = c.customer_id
WHERE o.order_status = 'COMPLETED';

## Gold Layer – Materialized View

In [ ]:
CREATE OR REFRESH MATERIALIZED VIEW gold_sales_by_country
COMMENT 'Revenue and order metrics by country (CDC-driven)'
AS
SELECT
  country,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(total_amount) AS revenue,
  AVG(total_amount) AS avg_order_value
FROM silver_orders_enriched
GROUP BY country;